In [ ]:
# Parameters
dayobs = 20251129

## Imports

In [ ]:
import asyncio
import lsst_efd_client

import astropy.units as u
from astropy.coordinates import get_sun, AltAz, EarthLocation
from astropy.time import Time, TimeDelta
from datetime import datetime, timezone, timedelta
import galsim
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytz
import sys
import matplotlib.dates as mdates

%matplotlib inline

In [ ]:
from lsst_efd_client import EfdClient
from typing import List

from lsst.summit.utils.efdUtils import (
    getEfdData,
    getDayObsEndTime,
    getDayObsForTime,
    getDayObsStartTime,
    makeEfdClient,
)

import matplotlib as mpl
efd_client = makeEfdClient()

In [ ]:
from lsst.summit.utils import (
    ConsDbClient,
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)
import os

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

## Define functions

In [ ]:
def getPsfGradPerZernike(
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
    jmax: int = 22,
) -> np.ndarray:
    """Get the gradient of the PSF FWHM with respect to each Zernike.

    This function takes no positional arguments. All parameters must be passed
    by name (see the list of parameters below).

    Parameters
    ----------
    diameter : float, optional
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float, optional
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int, optional
        The minimum Noll index, inclusive. Must be >= 0. (the default is 4)
    jmax : int, optional
        The max Zernike Noll index, inclusive. Must be >= jmin.
        (the default is 22.)

    Returns
    -------
    np.ndarray
        Gradient of the PSF FWHM with respect to the corresponding Zernike.
        Units are arcsec / micron.

    Raises
    ------
    ValueError
        If jmin is negative or jmax is less than jmin
    """
    # Check jmin and jmax
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")
    if jmax < jmin:
        raise ValueError("jmax must be greater than jmin.")

    # Calculate the conversion factors
    conversion_factors = np.zeros(jmax + 1)
    for i in range(jmin, jmax + 1):
        # Set coefficients for this Noll index: coefs = [0, 0, ..., 1]
        # Note the first coefficient is Noll index 0, which does not exist and
        # is therefore always ignored by galsim
        coefs = [0] * i + [1]

        # Create the Zernike polynomial with these coefficients
        R_outer = diameter / 2
        R_inner = R_outer * obscuration
        Z = galsim.zernike.Zernike(coefs, R_outer=R_outer, R_inner=R_inner)

        # We can calculate the size of the PSF from the RMS of the gradient of
        # the wavefront. The gradient of the wavefront perturbs photon paths.
        # The RMS quantifies the size of the collective perturbation.
        # If we expand the wavefront gradient in another series of Zernike
        # polynomials, we can exploit the orthonormality of the Zernikes to
        # calculate the RMS from the Zernike coefficients.
        rms_tilt = np.sqrt(np.sum(Z.gradX.coef**2 + Z.gradY.coef**2) / 2)

        # Convert to arcsec per micron
        rms_tilt = np.rad2deg(rms_tilt * 1e-6) * 3600

        # Convert rms -> fwhm
        fwhm_tilt = 2 * np.sqrt(2 * np.log(2)) * rms_tilt

        # Save this conversion factor
        conversion_factors[i] = fwhm_tilt

    return conversion_factors[jmin:]


def convertZernikesToPsfWidth(
    zernikes: np.ndarray,
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
) -> np.ndarray:
    """Convert Zernike amplitudes to quadrature contribution to the PSF FWHM.

    Parameters
    ----------
    zernikes : np.ndarray
        Zernike amplitudes (in microns), starting with Noll index `jmin`.
        Either a 1D array of zernike amplitudes, or a 2D array, where each row
        corresponds to a different set of amplitudes.
    diameter : float
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int
        The minimum Zernike Noll index, inclusive. Must be >= 0. The
        max Noll index is inferred from `jmin` and the length of `zernikes`.
        (the default is 4, which ignores piston, x & y offsets, and tilt.)

    Returns
    -------
    dFWHM: np.ndarray
        Quadrature contribution of each Zernike vector to the PSF FWHM
        (in arcseconds).

    Notes
    -----
    Converting Zernike amplitudes to their quadrature contributions to the PSF
    FWHM allows for easier physical interpretation of Zernike amplitudes and
    the performance of the AOS system.

    For example, image we have a true set of zernikes, [Z4, Z5, Z6], such that
    ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.2, 0.3] arcsecs.
    These Zernike perturbations increase the PSF FWHM by
    sqrt[(0.1)^2 + (-0.2)^2 + (0.3)^2] ~ 0.37 arcsecs.

    If the AOS perfectly corrects for these perturbations, the PSF FWHM will
    not increase in size. However, imagine the AOS estimates zernikes, such
    that ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.3, 0.4] arcsecs.
    These estimated Zernikes, do not exactly match the true Zernikes above.
    Therefore, the post-correction PSF will still be degraded with respect to
    the optimal PSF. In particular, the PSF FWHM will be increased by
    sqrt[(0.1 - 0.1)^2 + (-0.2 - (-0.3))^2 + (0.3 - 0.4)^2] ~ 0.14 arcsecs.

    This conversion depends on a linear approximation that begins to break down
    for RSS(dFWHM) > 0.20 arcsecs. Beyond this point, the approximation tends
    to overestimate the PSF degradation. In other words, if
    sqrt(sum( dFWHM^2 )) > 0.20 arcsec, it is likely that dFWHM is
    over-estimated. However, the point beyond which this breakdown begins
    (and whether the approximation over- or under-estimates dFWHM) can change,
    depending on which Zernikes have large amplitudes. In general, if you have
    large Zernike amplitudes, proceed with caution!
    Note that if the amplitudes Z_est and Z_true are large, this is okay, as
    long as |Z_est - Z_true| is small.

    For a notebook demonstrating where the approximation breaks down:
    https://gist.github.com/jfcrenshaw/24056516cfa3ce0237e39507674a43e1

    Raises
    ------
    ValueError
        If jmin is negative
    """
    # Check jmin
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")

    # Calculate jmax from jmin and the length of the zernike array
    jmax = jmin + np.array(zernikes).shape[-1] - 1

    # Calculate the conversion factors for each zernike
    conversion_factors = getPsfGradPerZernike(
        jmin=jmin,
        jmax=jmax,
        diameter=diameter,
        obscuration=obscuration,
    )

    # Convert the Zernike amplitudes from microns to their quadrature
    # contribution to the PSF FWHM
    dFWHM = conversion_factors * zernikes

    return dFWHM

In [ ]:
def convert_dataframe_to_timezone(
    df: pd.DataFrame, target_tz: str = "America/Santiago", timestamp_column: str = "timestamp"
) -> pd.DataFrame:
    """Convert a timestamp column or index to be timezone-aware.

    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame.
    target_tz : str
        The target timezone (e.g., "America/Santiago").
    timestamp_column : str
        Name of the timestamp column, if index is not already datetime-like.

    Returns
    -------
    pd.DataFrame
        A copy of the DataFrame with a localized DatetimeIndex.
    """
    if df.empty:
        return df.copy()

    df = df.copy()

    if isinstance(df.index, pd.DatetimeIndex):
        df.index = (
            df.index.tz_convert(target_tz)
            if df.index.tzinfo
            else df.index.tz_localize("UTC").tz_convert(target_tz)
        )
    elif timestamp_column in df.columns:
        df[timestamp_column] = pd.to_datetime(df[timestamp_column], utc=True, format='ISO8601')
        df.set_index(timestamp_column, inplace=True)
        df.index = df.index.tz_convert(target_tz)
    else:
        raise ValueError("DataFrame has no datetime index and no valid timestamp column.")

    return df


async def query_efd_grouped(client: lsst_efd_client.EfdClient, start_date: Time, end_date: Time) -> pd.DataFrame:
    topic_name = "lsst.sal.MTM1M3TS.thermalData"
    fields = [f"mean(absoluteTemperature{i})" for i in range(96)]
    query = client.build_time_range_query(
        topic_name,
        fields,
        start_date,
        end_date,
    )
    return await client._do_query(query + " GROUP BY time(1m)")


async def query_m1m3_ess(
    client: lsst_efd_client.EfdClient, start_date: Time, end_date: Time
) -> list[pd.DataFrame]:
    data_frames = []

    ess_start = 114
    ess_end = 117
    num_channels = (16, 16, 16, 16, 16, 15)
    for ess_index in range(ess_start, ess_end + 1):
        topic_name = "lsst.sal.ESS.temperature"
        for i in range(1, 7):
            fields = [f"mean(temperatureItem{i}) AS ch{i+1}" for i in range(num_channels[i - 1])]
            query = client.build_time_range_query(topic_name, fields, start_date, end_date, index=ess_index)
            query += f" AND salIndex = {ess_index}"
            query += f" AND sensorName =~ /{i}\\/6/ GROUP BY time(1m)"
            table = await client._do_query(query)
            table = table.add_prefix(f"ess{ess_index}_{i}_")
            data_frames.append(table)

    return pd.concat(data_frames, axis=1)
  
async def query_tma_truss_ess(
    client: lsst_efd_client.EfdClient, start_date: Time, end_date: Time
) -> list[pd.DataFrame]:
    data_frames = []

    topic_name = "lsst.sal.ESS.temperature"
    ess_index = 2
    name_dict = {6:'tma_truss_+x_+y', 7:'tma_truss_-x_-y'}
    for i in range(6, 8):
        fields = [f"mean(temperatureItem{i}) AS ch{i}"]
        query = client.build_time_range_query(topic_name, fields, start_date, end_date, index=ess_index)
        query += f" GROUP BY time(1m)"
        table = await client._do_query(query)
        table.rename(columns={f'ch{i}': name_dict[i]}, inplace=True)
        # table = table.add_prefix(f"ess{ess_index}_{i}_")
        data_frames.append(table)

    return pd.concat(data_frames, axis=1)

async def query_m2(client: lsst_efd_client.EfdClient, start_date: Time, end_date: Time) -> pd.DataFrame:
    """Query the EFD for MTM2 temperature data.

    This function obtains temperature data from the MTM2.temperature topic.

    Parameters
    ----------
    client : lsst_efd_client.EfdClient
        The EFD client to use for querying.
    start_date : Time
        The start date for the query.
    end_date : Time
        The end date for the query.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the mean temperatures for rings 0-11 (excluding ring5)
        and ring5 as a separate column, averaged over 1-minute intervals.
    """
    topic_name = "lsst.sal.MTM2.temperature"
    fields = [f"mean(ring{i})" for i in range(12)]
    query = client.build_time_range_query(
        topic_name,
        fields,
        start_date,
        end_date,
    )
    df = await client._do_query(query + " GROUP BY time(1m)")
    
    # Separate ring5 from the other rings
    ring5_data = df["mean_5"]
    other_rings = df.drop(columns=["mean_5"])
    
    # Calculate average of all rings except ring5
    temperature_avg = other_rings.mean(axis=1)
    
    # Create result DataFrame with both columns
    result = pd.DataFrame({
        "temperature": temperature_avg,
        "ring5": ring5_data
    })
    
    return result


async def query_efd(start_date: Time, end_date: Time) -> dict:
    # Define a time window for the previous day
    # client = lsst_efd_client.EfdClient("summit_efd")
    client = lsst_efd_client.EfdClient("usdf_efd")

    # Collect weather station temperatures
    topic = "lsst.sal.ESS.temperature"
    fields = ["temperatureItem0"]
    temperature_outdoor = await client.select_time_series(topic, fields, start_date, end_date, index=301)
    temperature_indoor_camera = await client.select_time_series(topic, fields, start_date, end_date, index=111)
    temperature_indoor_m2 = await client.select_time_series(topic, fields, start_date, end_date, index=112)
    temperature_indoor_m1m3 = await client.select_time_series(topic, fields, start_date, end_date, index=113)
    
    # Collect wind speed
    topic ="lsst.sal.ESS.airFlow"
    fields=["speed"]
    wind = await client.select_time_series(topic, fields, start_date, end_date, index=301)
    
    # Collect M1M3TS temperatures
    temperature_mtm1m3ts = await query_m1m3_ess(client, start_date, end_date)

    # Collect MTM2 temperatures
    temperature_mtm2 = await query_m2(client, start_date, end_date)

    # Collect TMA truss temperatures
    temperature_tma_truss = await query_tma_truss_ess(client, start_date, end_date)

    # Collect top end temperatures
    topic = "lsst.sal.MTMount.topEndChiller"
    fields = ["ambientTemperatureSensor0502"]
    temperature_topend = await client.select_time_series(topic, fields, start_date, end_date)

    # Collect MTM1M3TS setpoints
    topic = "lsst.sal.MTM1M3TS.command_applySetpoints"
    fields = ["heatersSetpoint", "private_identity"]
    mtm1m3ts_setpoints = await client.select_time_series(topic, fields, start_date, end_date)

    # Collect top end setpoints
    topic = "lsst.sal.MTMount.command_setThermal"
    fields = ["topEndChillerSetpoint", "private_identity"]
    topend_setpoints = await client.select_time_series(topic, fields, start_date, end_date)

    # Collect HVAC setpoints
    topic = "lsst.sal.HVAC.command_configLowerAhu"
    fields = ["workingSetpoint", "private_identity"]
    hvac_setpoints = await client.select_time_series(topic, fields, start_date, end_date)

    # Is dome open?
    topic = "lsst.sal.MTDome.apertureShutter"
    fields = ["positionActual0", "positionActual1"]
    shutter = await client.select_time_series(topic, fields, start_date, end_date)

    temperature_outdoor = convert_dataframe_to_timezone(temperature_outdoor)
    temperature_indoor_camera = convert_dataframe_to_timezone(temperature_indoor_camera)
    temperature_indoor_m2 = convert_dataframe_to_timezone(temperature_indoor_m2)
    temperature_indoor_m1m3 = convert_dataframe_to_timezone(temperature_indoor_m1m3)
    
    temperature_mtm1m3ts = convert_dataframe_to_timezone(temperature_mtm1m3ts)
    temperature_mtm2 = convert_dataframe_to_timezone(temperature_mtm2)
    temperature_tma_truss = convert_dataframe_to_timezone(temperature_tma_truss)
    temerature_topend = convert_dataframe_to_timezone(temperature_topend)
    mtm1m3ts_setpoints = convert_dataframe_to_timezone(mtm1m3ts_setpoints)
    topend_setpoints = convert_dataframe_to_timezone(topend_setpoints)
    hvac_setpoints = convert_dataframe_to_timezone(hvac_setpoints)
    shutter = convert_dataframe_to_timezone(shutter)
    wind=convert_dataframe_to_timezone(wind)

    return {
        "temperature_outdoor": temperature_outdoor,
        "temperature_indoor_camera": temperature_indoor_camera,
        "temperature_indoor_m2": temperature_indoor_m2,
        "temperature_indoor_m1m3": temperature_indoor_m1m3,
        "temperature_mtm1m3ts": temperature_mtm1m3ts,
        "temperature_mtm2": temperature_mtm2,
        "temperature_tma_truss": temperature_tma_truss,
        "temperature_topend": temperature_topend,
        "mtm1m3ts_setpoints": mtm1m3ts_setpoints,
        "topend_setpoints": topend_setpoints,
        "hvac_setpoints": hvac_setpoints,
        "shutter": shutter,
        "wind": wind,
    }


def get_shutter_closed_intervals(shutter: pd.DataFrame) -> list[tuple[pd.Timestamp | None, pd.Timestamp | None]]:
    """Find time intervals when the shutter was closed.

    Parameters
    ----------
    shutter : pd.DataFrame
        EFD archival data containing positionActual0 and positionActual1
        telemetry.

    Returns
    -------
    list[tuple[pd.Timestamp | None, pd.Timestamp | None]]
        The beginning and end of each interval in the dataset during which the
        dome was closed.
    """
    closed_intervals = []
    current_start = None

    shutter_is_open = (shutter["positionActual0"] >= 50) | (shutter["positionActual1"] >= 50)

    for time, is_open in shutter_is_open.items():
        if not is_open:
            if current_start is None:
                current_start = time  # mark the beginning of a closed period
        else:
            if current_start is not None:
                closed_intervals.append((current_start, time))
                current_start = None

    # Handle case where dome was closed until the end
    if current_start is not None:
        closed_intervals.append((current_start, shutter_is_open.index[-1]))

    return closed_intervals

def get_sv_intervals(cons_db_data: pd.DataFrame) -> list[tuple[pd.Timestamp | None, pd.Timestamp | None]]:
    """Find time intervals when sv observations were taken.

    Parameters
    ----------
    cons_db_data : pd.DataFrame
        ConsDB archival data containing positionActual0 and positionActual1
        telemetry.

    Returns
    -------
    list[tuple[pd.Timestamp | None, pd.Timestamp | None]]
        The beginning and end of each interval in the dataset during which the
        sv data was taken.
    """
    sv_intervals = []
    current_start = None

    sv_data = cons_db_data["science_program"] == "BLOCK-365"

    for time, is_sv in sv_data.items():
        if is_sv is True:
            if current_start is None:
                current_start = time  # mark the beginning of an SV block
        else:
            if current_start is not None:
                sv_intervals.append((current_start, time))
                current_start = None

    # Handle case where sv was run until end of night
    if current_start is not None:
        sv_intervals.append((current_start, sv_data.index[-1]))

    return sv_intervals


def find_sun_altitude_crossings(start_time: Time, end_time: Time, altitude_deg: float, rising: bool) -> list:
    """
    Return a list of localized datetimes when the sun crosses a given altitude
    (e.g., -18 for twilight, 0 for sunrise/sunset) between start_time and end_time.

    Parameters
    ----------
    start_time : Time
        Astropy start time.
    end_time : Time
        Astropy end time.
    altitude_deg : float
        Target sun altitude in degrees.
    rising : bool
        True for rising (upward crossing), False for setting (downward).

    Returns
    -------
    list[datetime]
        Localized CLT datetimes when the sun crosses the given altitude.
    """
    location = EarthLocation.of_site("Cerro Pachon")
    clt = pytz.timezone("America/Santiago")

    step_seconds = 60
    n_steps = int((end_time - start_time).sec / step_seconds) + 1
    times = start_time + TimeDelta(np.arange(n_steps) * step_seconds, format='sec')

    altaz = AltAz(obstime=times, location=location)
    sun_alt = get_sun(times).transform_to(altaz).alt.deg
    times_clt = times.to_datetime(timezone=clt)

    crossings = []
    for i in range(1, len(sun_alt)):
        prev_alt = sun_alt[i - 1]
        curr_alt = sun_alt[i]
        if rising and prev_alt < altitude_deg <= curr_alt:
            crossings.append(times_clt[i])
        elif not rising and prev_alt > altitude_deg >= curr_alt:
            crossings.append(times_clt[i])

    return crossings


In [ ]:
def plot_temperature_mtm1m3ts(ax, temperature_mtm1m3ts):
    mean_temp = temperature_mtm1m3ts.mean(axis=1)
    min_temp = temperature_mtm1m3ts.min(axis=1)
    max_temp = temperature_mtm1m3ts.max(axis=1)

    lower_pct = temperature_mtm1m3ts.quantile(0.1587, axis=1)
    upper_pct = temperature_mtm1m3ts.quantile(0.8413, axis=1)

    # Min–max band (light teal)
    ax.fill_between(
        temperature_mtm1m3ts.index,
        min_temp,
        max_temp,
        color="#c7ecee",  # light teal
        alpha=0.6,
        zorder=1,
    )

    # ±1 std band (medium teal)
    ax.fill_between(
        temperature_mtm1m3ts.index,
        lower_pct,
        upper_pct,
        color="#76d7c4",  # medium teal
        alpha=0.8,
        zorder=2,
    )

    # Mean line (deep teal)
    ax.plot(
        temperature_mtm1m3ts.index,
        mean_temp,
        color="#117864",  # deep teal
        label="M1M3 Temperature",
        zorder=3,
    )

    # print(min(min_temp), max(max_temp), min(lower_pct), max(upper_pct))

band_colors = {
    "u": "#0c71ff",
    "g": "#49be61",
    "r": "#c61c00",
    "i": "#ffc200",
    "z": "#f341a2",
    "y": "#5d0000",
}

def annotate_bands(data: pd.DataFrame, ax: plt.Axes) -> None:
    # Get the axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # Use actual datetime index instead of integer positions
    timestamps = data.index.values
    
    if len(timestamps) == 0:
        return
        
    # Create time intervals for each band
    for i, (timestamp, band) in enumerate(zip(timestamps, data['band'].values)):
        # Calculate width for each band segment
        if i < len(timestamps) - 1:
            next_timestamp = timestamps[i + 1]
            width = (next_timestamp - timestamp)
        else:
            # For last point, use same width as previous
            if i > 0:
                prev_timestamp = timestamps[i - 1]
                width = (timestamp - prev_timestamp)
        
        # Plot band bar
        ax.barh(
            ylim[0] + (ylim[1] - ylim[0]) * 0.02,  # slightly above bottom
            width,
            left=timestamp,
            height=(ylim[1] - ylim[0]) * 0.03,  # 3% of plot height
            color=band_colors[band[0]],
            alpha=0.8
        )

    # Restore the axis limits
    ax.set_ylim(ylim)
    ax.set_xlim(xlim)

    # Add unique bands to legend
    unique_bands = data['band'].str[0].unique()  # Get first character of each band
    
    for band_char in sorted(unique_bands):
        ax.scatter([], [], 
                  c=band_colors[band_char], 
                  s=100, 
                  marker='s', 
                  label=f'Band {band_char}')
    
def draw_plot(
    start_date: Time,
    end_date: Time,
    *,
    temperature_outdoor: pd.DataFrame,
    temperature_indoor_camera: pd.DataFrame,
    temperature_indoor_m2: pd.DataFrame,
    temperature_indoor_m1m3: pd.DataFrame,
    temperature_mtm1m3ts: pd.DataFrame,
    temperature_mtm2: pd.DataFrame,
    temperature_tma_truss: pd.DataFrame,
    temperature_topend: pd.DataFrame,
    mtm1m3ts_setpoints: pd.DataFrame,
    topend_setpoints: pd.DataFrame,
    hvac_setpoints: pd.DataFrame,
    shutter: pd.DataFrame,
    wind: pd.DataFrame,
):

    # Create subplots with shared x-axis
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12), sharex=True)

    closed_intervals = get_shutter_closed_intervals(shutter)

    # Plot background shading for closed periods on both subplots
    for start, end in closed_intervals:
        ax1.axvspan(start, end, facecolor="lightgray", alpha=0.3, zorder=0, label="_dome_closed")
        ax2.axvspan(start, end, facecolor="lightgray", alpha=0.3, zorder=0, label="_dome_closed")

    if closed_intervals:
        ax1.axvspan(
            closed_intervals[0][0],
            closed_intervals[0][0],  # dummy zero-width span just for legend
            facecolor="lightgray",
            alpha=0.3,
            label="Dome Closed",
        )

    # --- FIRST SUBPLOT: Temperature data ---
    # Line plots: outdoor and indoor temperatures
    ax1.plot(
        temperature_outdoor.index,
        temperature_outdoor["temperatureItem0"],
        label="Outdoor Temp (Index 301)",
    )
    ax1.plot(
        temperature_indoor_camera.index,
        temperature_indoor_camera["temperatureItem0"],
        label="Indoor Temp near camera (Index 111)",
        linestyle='--'
    )
    ax1.plot(
        temperature_indoor_m2.index,
        temperature_indoor_m2["temperatureItem0"],
        label="Indoor Temp near M2(Index 112)",
        linestyle='--'
    )
    ax1.plot(
        temperature_indoor_m1m3.index,
        temperature_indoor_m1m3["temperatureItem0"],
        label="Indoor Temp near M1M3 (Index 113)",
        linestyle='--'
    )
    plot_temperature_mtm1m3ts(ax1, temperature_mtm1m3ts)

    ax1.plot(
        temperature_mtm2.index,
        temperature_mtm2["temperature"],
        label="M2 Temp (exc. Ring5)",
        color="magenta",
    )

    ax1.plot(
        temperature_mtm2.index,
        temperature_mtm2["ring5"],
        label="M2 Ambient Temp (Ring5)",
        color="crimson",
    )   

    ax1.plot(
        temperature_topend.index,
        temperature_topend["ambientTemperatureSensor0502"],
        label="Top End Temperature (Index 0502)",
        color="sienna",
    )

    if not temperature_tma_truss.empty:
        ax1.plot(
            temperature_tma_truss.index,
            temperature_tma_truss['tma_truss_+x_+y'],
            label='TMA Truss Temperature (+x,+y)',
            color='turquoise'
        )
    
        ax1.plot(
            temperature_tma_truss.index,
            temperature_tma_truss['tma_truss_-x_-y'],
            label='TMA Truss Temperature (-x,-y)',
            color='navy'
        )
    
    # Top End Setpoints
    if not topend_setpoints.empty:
        topend = topend_setpoints.copy()
        topend["adjusted"] = (
            topend["topEndChillerSetpoint"]
        )

        for identity in topend["private_identity"].unique():
            mask = topend["private_identity"] == identity
            ax1.scatter(
                topend.index[mask],
                topend["adjusted"][mask],
                label="Top End Setpoint" if identity == "EAS" else None,
                color="tab:red" if identity == "EAS" else "gray",
                marker="x",
                zorder=4,
            )

    # HVAC Setpoints (if present)
    if not hvac_setpoints.empty:
        hvac = hvac_setpoints.copy()
        hvac["adjusted"] = hvac["workingSetpoint"]
        ax1.plot(hvac.index, hvac["adjusted"], linestyle=":", color="black", alpha=0.4)

        for identity in hvac["private_identity"].unique():
            mask = hvac["private_identity"] == identity
            ax1.scatter(
                hvac.index[mask],
                hvac["adjusted"][mask],
                label="HVAC Setpoint" if identity == "EAS" else None,
                color="tab:purple" if identity == "EAS" else "gray",
                marker="s",
            )

    # MTM1M3TS Setpoints
    if not mtm1m3ts_setpoints.empty:
        mtm1 = mtm1m3ts_setpoints.copy()
        mtm1["adjusted"] = mtm1["heatersSetpoint"] # no adjustment. Heaters set point should track ~1 deg below ambient
        ax1.plot(mtm1.index, mtm1["adjusted"], linestyle=":", color="black", alpha=0.4, zorder=2)

        for identity in mtm1["private_identity"].unique():
            mask = mtm1["private_identity"] == identity
            ax1.scatter(
                mtm1.index[mask],
                mtm1["adjusted"][mask],
                label="M1M3 Setpoint" if identity == "EAS" else None,
                color="tab:green" if identity == "EAS" else "gray",
                marker="o",
                s=100,  # larger size
                zorder=5,  # ensure it's on top
                alpha=0.5,
            )

    twilight_ends = find_sun_altitude_crossings(start_date, end_date, altitude_deg=-18, rising=False)
    sunrises = find_sun_altitude_crossings(start_date, end_date, altitude_deg=0, rising=True)

    if twilight_ends:
        ax1.axvline(twilight_ends[0], color="black", linestyle="--", linewidth=2.5, label="Twilight End")
        ax2.axvline(twilight_ends[0], color="black", linestyle="--", linewidth=2.5, label="Twilight End")
    if sunrises:
        ax1.axvline(sunrises[0], color="orange", linestyle="--", linewidth=2.5, label="Sunrise")
        ax2.axvline(sunrises[0], color="orange", linestyle="--", linewidth=2.5, label="Sunrise")

    # First subplot formatting
    ax1.set_ylabel("Temperature (°C)")
    ax1.legend(frameon=True, framealpha=1.0, facecolor='white', ncol=2)

    # --- SECOND SUBPLOT: Image Quality data ---
    ax2.plot(cdb_sub_table.index, cdb_sub_table['fwhm_zenith_500nm'],  label='fwhm_zenith_500nm')
    ax2.plot(cdb_sub_table.index, cdb_sub_table['dimm'], label='dimm')
    ax2.plot(cdb_sub_table.index, cdb_sub_table['aos_fwhm'], label='aos_fwhm')
    ax2.set_ylabel('FWHM (arcsec)')
    ax2.set_title('Image Quality Measurements')

    # Add shading for SV data
    sv_intervals = get_sv_intervals(cdb_sub_table)

    # Plot background shading for closed periods on both subplots
    for start, end in sv_intervals:
        ax2.axvspan(start, end, facecolor="pink", alpha=0.3, zorder=0, label="_sv_observations")

    if sv_intervals:
        ax2.axvspan(
            sv_intervals[0][0],
            sv_intervals[0][0],  # dummy zero-width span just for legend
            facecolor="pink",
            alpha=0.3,
            label="SV Observations",
        )

    annotate_bands(cdb_sub_table, ax2)
    
    ax2.legend(frameon=True, framealpha=1.0, facecolor='white')

    # Format datetime x-axis ticks (only on bottom subplot)
    locator = mdates.AutoDateLocator()
    formatter = mdates.ConciseDateFormatter(locator)
    ax2.xaxis.set_major_locator(locator)
    ax2.xaxis.set_major_formatter(formatter)
    ax2.set_xlabel("Time (UTC)")

    # Add start date to title (on top subplot)
    start_dt_str = start_date.to_datetime().strftime("%Y-%m-%d")
    ax1.set_title(f"EAS Summary – {start_dt_str}")
    
    fig.tight_layout()
    return fig, (ax1, ax2)




## Find the time took to stabilize after dome opening

In [ ]:
async def query_efd_shutter(start_date: Time, end_date: Time) -> dict:
    # Define a time window for the previous day
    # client = lsst_efd_client.EfdClient("summit_efd")
    client = lsst_efd_client.EfdClient("usdf_efd")
    # Is dome open?
    topic = "lsst.sal.MTDome.apertureShutter"
    fields = ["positionActual0", "positionActual1"]
    shutter = await client.select_time_series(topic, fields, start_date, end_date)

    shutter = convert_dataframe_to_timezone(shutter)

    return {
        "shutter": shutter,
    }

In [ ]:
def check_dome_open(
    start_date: Time,
    end_date: Time,
    *,
    shutter: pd.DataFrame,
):
    if np.all(shutter['positionActual0'])<1:
        print('never opened')
        t1=t2=np.nan
    for i in np.arange(len(shutter)):
        if not i==0:
            if shutter['positionActual0'][i]<1:
                if shutter['positionActual0'][i-1]<1:
                    pass
                else:
                    print('closed')
            else:
                if shutter['positionActual0'][i-1]<shutter['positionActual0'][i]:
                    if shutter['positionActual0'][i]<np.max(shutter['positionActual0']):
                        if shutter['positionActual0'][i-1]<1:
                            print('opened at', shutter.index[i], shutter['positionActual0'][i])
                            t1=shutter.index[i]
                    else:
                        print('fully opened at', shutter.index[i], shutter['positionActual0'][i])
                        t2=shutter.index[i]
    return {
        t1,t2,
    }
    #closed_intervals = get_shutter_closed_intervals(shutter)

    #if closed_intervals:
    #    print(closed_intervals)

In [ ]:
# Parameters
year = int(str(dayobs)[:4])
month = int(str(dayobs)[4:6])
day = int(str(dayobs)[6:])
start_dt = datetime(year, month, day, 22, 0, 0, tzinfo=timezone.utc)
start = Time(start_dt)

end_dt = datetime(year, month, day+1, 8, 0, 0, tzinfo=timezone.utc)
end = Time(end_dt)

In [ ]:
efd_results = await asyncio.create_task(query_efd(start, end))
efd_shutter_results = await asyncio.create_task(query_efd_shutter(start, end))

In [ ]:
def draw_plot_deltat(
    start_date: Time,
    end_date: Time,
    *,
    temperature_outdoor: pd.DataFrame,
    temperature_indoor_camera: pd.DataFrame,
    temperature_indoor_m2: pd.DataFrame,
    temperature_indoor_m1m3: pd.DataFrame,
    temperature_mtm1m3ts: pd.DataFrame,
    temperature_mtm2: pd.DataFrame,
    temperature_tma_truss: pd.DataFrame,
    temperature_topend: pd.DataFrame,
    mtm1m3ts_setpoints: pd.DataFrame,
    topend_setpoints: pd.DataFrame,
    hvac_setpoints: pd.DataFrame,
    shutter: pd.DataFrame,
    wind: pd.DataFrame,
):

    # Create subplots with shared x-axis
    fig, ax1 = plt.subplots(1, 1, figsize=(15, 6), sharex=True)

    closed_intervals = get_shutter_closed_intervals(shutter)

    # Plot background shading for closed periods on both subplots
    for start, end in closed_intervals:
        ax1.axvspan(start, end, facecolor="lightgray", alpha=0.3, zorder=0, label="_dome_closed")

    if closed_intervals:
        ax1.axvspan(
            closed_intervals[0][0],
            closed_intervals[0][0],  # dummy zero-width span just for legend
            facecolor="lightgray",
            alpha=0.3,
            label="Dome Closed",
        )
    t1,t2=check_dome_open(start, end, **efd_shutter_results)
    print(t1,t2)
    if t2:
        ax1.axvline(t1, color="black", linestyle="--", linewidth=2.5, label="dome open starts")
        ax1.axvline(t2, color="black", linestyle="--", linewidth=2.5, label="dome open completes")
    # --- FIRST SUBPLOT: Temperature data ---
    # Line plots: outdoor and indoor temperatures

    ax1.plot(
        temperature_outdoor.index,
        temperature_outdoor["temperatureItem0"]-temperature_outdoor["temperatureItem0"][-1],
        label="Outdoor Temp (Index 301)",
    )
    ax1.plot(
        temperature_indoor_camera.index,
        temperature_indoor_camera["temperatureItem0"]-temperature_indoor_camera["temperatureItem0"][-1],
        label="Indoor Temp near camera (Index 111)",
    )
    ax1.plot(
        temperature_indoor_m2.index,
        temperature_indoor_m2["temperatureItem0"]-temperature_indoor_m2["temperatureItem0"][-1],
        label="Indoor Temp near M2 (Index 112)",
    )
    ax1.plot(
        temperature_indoor_m1m3.index,
        temperature_indoor_m1m3["temperatureItem0"]-temperature_indoor_m1m3["temperatureItem0"][-1],
        label="Indoor Temp near M1M3 (Index 113)",
    )

    #plot_temperature_mtm1m3ts(ax1, temperature_mtm1m3ts)

    ax1.plot(
        temperature_mtm2.index,
        temperature_mtm2["temperature"]-temperature_mtm2["temperature"][-1],
        label="M2 Temp (exc. Ring5)",
        color="magenta",
    )

    ax1.plot(
        temperature_mtm2.index,
        temperature_mtm2["ring5"]-temperature_mtm2["ring5"][-1],
        label="M2 Ambient Temp (Ring5)",
        color="crimson",
    )   

    ax1.plot(
        temperature_topend.index,
        temperature_topend["ambientTemperatureSensor0502"]-temperature_topend["ambientTemperatureSensor0502"][-1],
        label="Top End Temperature (Index 0502)",
        color="sienna",
    )

    if not temperature_tma_truss.empty:
        ax1.plot(
            temperature_tma_truss.index,
            temperature_tma_truss['tma_truss_+x_+y']-temperature_tma_truss['tma_truss_+x_+y'][-1],
            label='TMA Truss Temperature (+x,+y)',
            color='turquoise'
        )
    
        ax1.plot(
            temperature_tma_truss.index,
            temperature_tma_truss['tma_truss_-x_-y']-temperature_tma_truss['tma_truss_-x_-y'][-1],
            label='TMA Truss Temperature (-x,-y)',
            color='navy'
        )
    


    # Add start date to title (on top subplot)
    start_dt_str = start_date.to_datetime().strftime("%Y-%m-%d")
    ax1.set_title(f"EAS Summary – {start_dt_str}")
    
    ax1.legend(frameon=True, framealpha=1.0, facecolor='white', ncol=2)
    fig.tight_layout()
    return fig, ax1




In [ ]:
async def single_sampled_query(
    client: EfdClient, 
    topic: str,
    columns: List[str],
    begin: Time,
    end: Time, 
    sal_index: int | None=None,
    sampling: str="10s"
    ):

    query = client.build_time_range_query(topic, columns, begin, end, index=sal_index)
    query += f" GROUP BY time({sampling})"

    df = await client.influx_client.query(query)
    df.dropna(inplace=True)

    return df
    

async def query_and_concat_data(_efd_client: EfdClient, _day_obs: int, start, end, sampling: str="10s"):

    # Let's start with the time stamps
    start_time=start
    end_time=end
    start_day_obs_time = getDayObsStartTime(_day_obs)
    end_day_obs_time = getDayObsEndTime(_day_obs)

    start_night_time = find_sun_altitude_crossings(
        start_day_obs_time, end_day_obs_time, altitude_deg=-18, rising=False)[0]
    end_night_time = find_sun_altitude_crossings(
        start_day_obs_time, end_day_obs_time, altitude_deg=-18, rising=True)[0]

    start_night_time = Time(start_night_time, scale="utc")
    end_night_time = Time(end_night_time, scale="utc")

    dfs = []

    # We will keep queries separated to it is easier to refactor later
    
    queries = [
        # ESS:111 Camera air Temperature 
        dict(
            topic="lsst.sal.ESS.temperature",
            columns=[f"mean(temperatureItem0) as temperature_ESS111"],    
            sal_index=111
        ),
        # ESS:112 M2 air Temperature 
        dict(
            topic="lsst.sal.ESS.temperature",
            columns=[f"mean(temperatureItem0) as temperature_ESS112"],    
            sal_index=112
        ),
        # ESS:113 M1M3 air Temperature 
        dict(
            topic="lsst.sal.ESS.temperature",
            columns=[f"mean(temperatureItem0) as temperature_ESS113"],    
            sal_index=113
        ),
        # ESS:301 Outside Temperature
        dict(
            topic="lsst.sal.ESS.temperature",
            columns=[f"mean(temperatureItem0) as temperature_ESS301"],
            sal_index=301
        ),

        #TEA
        dict(
            topic="lsst.sal.MTMount.topEndChiller",
            columns=[f"mean(ambientTemperatureSensor0502) as temperature_TEA502"],
            sal_index=None
        ),
        
        # Telescope elevation 
        dict(
            topic="lsst.sal.MTMount.elevation",
            columns=["mean(actualPosition) as tel_elevation"],
            sal_index=None
        ),
        
        # Wind speed
        dict(
            topic="lsst.sal.ESS.airFlow",
            columns=[f"mean(speed) as wind_speed"],
            sal_index=301
        ),

        #M1M3TS setpoints
        dict(
            topic="lsst.sal.MTM1M3TS.command_applySetpoints",
            columns=["mean(heatersSetpoint) as heater_setpoint"],
            sal_index=None
        ),
        #M2 ring3
        dict(
            topic="lsst.sal.MTM2.temperature",
            columns=["mean(ring3) as m2_glass_temp"],
            sal_index=None
        ),
        #M1M3 glass
        dict(
            topic="lsst.sal.ESS.temperature",
            columns=["mean(temperatureItem11) as m1m3_glass_temp"],
            sal_index=115
        ),
        #wind direction
        dict(
            topic="lsst.sal.ESS.airFlow",
            columns=["mean(direction) as wind_direction"],
            sal_index=301
        ),
        #dome az position
        dict(
            topic="lsst.sal.MTDome.azimuth",
            columns=["mean(positionActual) as dome_az"],
            sal_index=None
        ),
        #louver
        dict(
            topic="lsst.sal.MTDome.louvers",
            columns=["mean(positionActual11) as louverE1", "mean(positionActual12) as louverE2", "mean(positionActual20) as louverH1", "mean(positionActual21) as louverH2", "mean(positionActual2) as louverB1", "mean(positionActual29) as louverM1"],
            sal_index=None
        ),
        #shutter
        dict(
            topic = "lsst.sal.MTDome.apertureShutter",
            columns = ["mean(positionActual0) as shutter"],
            sal_index=None
        ),
        #107
        dict(
            topic="lsst.sal.ESS.temperature",
            columns=[f"mean(temperatureItem0) as ESS107temp"],
            sal_index=107
        ),
        #humidity
        dict(
            topic="lsst.sal.ESS.relativeHumidity",
            columns=[f"mean(relativeHumidityItem) as humidity"],
            sal_index=301
        ),
    ]

    
    # Collect M1M3TS temperatures
    #temperature_mtm1m3ts = await query_m1m3_ess(client, start_date, end_date)

    # Collect MTM2 temperatures
    #temperature_mtm2 = await query_m2(client, start_date, end_date)

    # Collect TMA truss temperatures
    #temperature_tma_truss = await query_tma_truss_ess(client, start_date, end_date)

    # Collect top end setpoints
    #topic = "lsst.sal.MTMount.command_setThermal"
    #fields = ["topEndChillerSetpoint", "private_identity"]
    #topend_setpoints = await client.select_time_series(topic, fields, start_date, end_date)

    # Collect HVAC setpoints
    #topic = "lsst.sal.HVAC.command_configLowerAhu"
    #fields = ["workingSetpoint", "private_identity"]
    #hvac_setpoints = await client.select_time_series(topic, fields, start_date, end_date)


    # Run the queries
    for q in queries:
        temp_df = await single_sampled_query(
            _efd_client,
            q["topic"],
            q["columns"],
            #start_night_time#,
            #end_night_time,
            start_time,
            end_time,
            sal_index=q["sal_index"]
        )
        dfs.append(temp_df)
            
    # Concatenate everything together
    output_df = pd.concat(dfs, axis=1)
    return output_df

In [ ]:
async def louver_statistics(_efd_client: EfdClient, _day_obs: int, start, end, sampling: str="10s"):
    start_time=start
    end_time=end
    start_day_obs_time = getDayObsStartTime(_day_obs)
    end_day_obs_time = getDayObsEndTime(_day_obs)

    start_night_time = find_sun_altitude_crossings(
        start_day_obs_time, end_day_obs_time, altitude_deg=-18, rising=False)[0]
    end_night_time = find_sun_altitude_crossings(
        start_day_obs_time, end_day_obs_time, altitude_deg=-18, rising=True)[0]

    start_night_time = Time(start_night_time, scale="utc")
    end_night_time = Time(end_night_time, scale="utc")

    dfs = []
    queries = [
        #louver
            dict(
                topic="lsst.sal.MTDome.louvers",
                columns=["mean(positionActual11) as louverE1", "mean(positionActual12) as louverE2", "mean(positionActual20) as louverH1", "mean(positionActual21) as louverH2", "mean(positionActual2) as louverB1", "mean(positionActual29) as louverM1"],
                sal_index=None
            ),
    ]
    for q in queries:
        temp_df = await single_sampled_query(
            _efd_client,
            q["topic"],
            q["columns"],
            #start_night_time#,
            #end_night_time,
            start_time,
            end_time,
            sal_index=q["sal_index"]
        )
        dfs.append(temp_df)
            
    # Concatenate everything together
    output_df = pd.concat(dfs, axis=1)
    return output_df

In [ ]:
dayobs = 20251115
year = int(str(dayobs)[:4])
month = int(str(dayobs)[4:6])
day = int(str(dayobs)[6:])
start_dt = datetime(year, month, day, 19, 0, 0, tzinfo=timezone.utc)
#start_dt = datetime(year, month, day, 1, 0, 0, tzinfo=timezone.utc)

start = Time(start_dt)

end_dt = datetime(year, month, day+1, 8, 0, 0, tzinfo=timezone.utc)
#end_dt = datetime(year, month, day, 5, 0, 0, tzinfo=timezone.utc)

end = Time(end_dt)

In [ ]:
louver_stat= await louver_statistics(efd_client, dayobs, start, end)
louver_stat

In [ ]:
df = await query_and_concat_data(efd_client, dayobs, start, end)
#df

In [ ]:
os.environ["no_proxy"] += ",.consdb"
consdb_url = 'http://consdb-pq.consdb:8080/consdb'
cdb_client = ConsDbClient(consdb_url)

In [ ]:
query = f"""
    SELECT
    e.airmass AS airmass,
    e.dimm_seeing AS dimm,
    e.altitude AS elevation,
    e.azimuth AS azimuth,
    e.exposure_id AS visit_id,
    e.physical_filter as band,
    e.day_obs AS day_obs,
    e.exp_midpt AS time,
    e.dimm_seeing AS seeing,
    e.science_program AS science_program,
    ccdvisit1_quicklook.psf_sigma,
    ccdvisit1_quicklook.z4,
    ccdvisit1_quicklook.z5,
    ccdvisit1_quicklook.z6,
    ccdvisit1_quicklook.z7,
    ccdvisit1_quicklook.z8,
    ccdvisit1_quicklook.z9,
    ccdvisit1_quicklook.z10,
    ccdvisit1_quicklook.z11,
    ccdvisit1_quicklook.z12,
    ccdvisit1_quicklook.z13,
    ccdvisit1_quicklook.z14,
    ccdvisit1_quicklook.z15,
    ccdvisit1_quicklook.z16,
    ccdvisit1_quicklook.z17,
    ccdvisit1_quicklook.z18,
    ccdvisit1_quicklook.z19,
    ccdvisit1_quicklook.z20,
    ccdvisit1_quicklook.z21,
    ccdvisit1_quicklook.z22,
    ccdvisit1_quicklook.z23,
    ccdvisit1_quicklook.z24,
    ccdvisit1_quicklook.z25,
    ccdvisit1_quicklook.z26,
    ccdvisit1_quicklook.z27,
    ccdvisit1_quicklook.z28,
    ccdvisit1.detector as detector,
    q.psf_sigma_median AS psf_fwhm,
    q.psf_sigma_min AS psf_fwhm_min,
    q.psf_sigma_max AS psf_fwhm_max,
    e.obs_end,
    e.obs_start
    FROM
    cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
    cdb_lsstcam.ccdvisit1 AS ccdvisit1,
    cdb_lsstcam.visit1 AS visit1,
    cdb_lsstcam.visit1_quicklook AS q,
    cdb_lsstcam.exposure AS e
    WHERE
    ccdvisit1.detector IN (191, 192, 195, 196, 199, 200, 203, 204)
    AND ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
    AND ccdvisit1.visit_id = visit1.visit_id
    AND ccdvisit1.visit_id = q.visit_id
    AND ccdvisit1.visit_id = e.exposure_id
    AND (e.img_type = 'science' or e.img_type = 'acq')
    AND e.exp_midpt > '{start.isot}'
    AND e.exp_midpt < '{end.isot}'
    AND e.airmass > 0
    AND e.band != 'none'
"""
cdb_table = cdb_client.query(query).to_pandas()

# Convert PSF sigma to FWHM
sig2fwhm = 2 * np.sqrt(2 * np.log(2))
pixel_scale = 0.2  # arcsec / pixel
cdb_table["psf_fwhm"] = cdb_table["psf_fwhm"] * sig2fwhm * pixel_scale

cdb_table["fwhm_zenith_500nm"] = [
    fwhm
    * getAirmassSeeingCorrection(airmass)
    * getBandpassSeeingCorrection(band)
    for fwhm, band, airmass in zip(
        cdb_table["psf_fwhm"], cdb_table["band"], cdb_table["airmass"]
    )
]

zernike_columns = [f"z{i}" for i in range(4, 29)]
cdb_table["zernikes"] = cdb_table[zernike_columns].apply(
    lambda row: np.array(row.fillna(0.0).values, dtype=float), axis=1
)
cdb_table["zernikes_fwhm"] = cdb_table["zernikes"].apply(
    convertZernikesToPsfWidth
)
cdb_table["aos_fwhm"] = 1.06 * np.log(
    1
    + np.sqrt(
        np.sum(np.square(np.vstack(cdb_table["zernikes_fwhm"].values)), axis=1)
    )
)

In [ ]:

# Group consdb data similarly to efd data
cdb_time_table = convert_dataframe_to_timezone(cdb_table, timestamp_column='obs_start')
cdb_sub_table = cdb_time_table[['dimm', 'aos_fwhm', 'fwhm_zenith_500nm', 'band', 'science_program']]
cdb_sub_table = cdb_sub_table.groupby('obs_start').agg({'dimm': 'mean', 'aos_fwhm': 'mean', 'fwhm_zenith_500nm': 'mean', 'band': 'first', 'science_program': 'first'})

In [ ]:
def get_shutter_closed_intervals(shutter: pd.DataFrame) -> list[tuple[pd.Timestamp | None, pd.Timestamp | None]]:
    """Find time intervals when the shutter was closed.

    Parameters
    ----------
    shutter : pd.DataFrame
        EFD archival data containing positionActual0 and positionActual1
        telemetry.

    Returns
    -------
    list[tuple[pd.Timestamp | None, pd.Timestamp | None]]
        The beginning and end of each interval in the dataset during which the
        dome was closed.
    """
    closed_intervals = []
    current_start = None

    shutter_is_open = (shutter >= 50)

    for time, is_open in shutter_is_open.items():
        if not is_open:
            if current_start is None:
                current_start = time  # mark the beginning of a closed period
        else:
            if current_start is not None:
                closed_intervals.append((current_start, time))
                current_start = None

    # Handle case where dome was closed until the end
    if current_start is not None:
        closed_intervals.append((current_start, shutter_is_open.index[-1]))

    return closed_intervals

In [ ]:
# From Matplotlib - Annotaded Heatmaps
# https://matplotlib.org/stable/gallery/images_contours_and_fields/image_annotated_heatmap.html#using-the-helper-function-code-style
def heatmap(data, row_labels, col_labels, ax=None,
            cbar_kw=None, cbarlabel="", **kwargs):
    """
    Create a heatmap from a numpy array and two lists of labels.

    Parameters
    ----------
    data
        A 2D numpy array of shape (M, N).
    row_labels
        A list or array of length M with the labels for the rows.
    col_labels
        A list or array of length N with the labels for the columns.
    ax
        A `matplotlib.axes.Axes` instance to which the heatmap is plotted.  If
        not provided, use current Axes or create a new one.  Optional.
    cbar_kw
        A dictionary with arguments to `matplotlib.Figure.colorbar`.  Optional.
    cbarlabel
        The label for the colorbar.  Optional.
    **kwargs
        All other arguments are forwarded to `imshow`.
    """

    if ax is None:
        ax = plt.gca()

    if cbar_kw is None:
        cbar_kw = {}

    # Plot the heatmap
    im = ax.imshow(data, **kwargs)

    # Create colorbar
    cbar = ax.figure.colorbar(im, ax=ax, **cbar_kw)
    cbar.ax.set_ylabel(cbarlabel, rotation=-90, va="bottom")

    # Show all ticks and label them with the respective list entries.
    ax.set_xticks(range(data.shape[1]), labels=col_labels,
                  rotation=-30, ha="right", rotation_mode="anchor")
    ax.set_yticks(range(data.shape[0]), labels=row_labels)

    # Let the horizontal axes labeling appear on top.
    ax.tick_params(top=True, bottom=False,
                   labeltop=True, labelbottom=False)

    # Turn spines off and create white grid.
    ax.spines[:].set_visible(False)

    ax.set_xticks(np.arange(data.shape[1]+1)-.5, minor=True)
    ax.set_yticks(np.arange(data.shape[0]+1)-.5, minor=True)
    ax.grid(which="minor", color="w", linestyle='-', linewidth=3)
    ax.tick_params(which="minor", bottom=False, left=False)

    return im, cbar


def annotate_heatmap(im, data=None, valfmt="{x:.2f}",
                     textcolors=("black", "white"),
                     threshold=None, **textkw):
    """
    A function to annotate a heatmap.

    Parameters
    ----------
    im
        The AxesImage to be labeled.
    data
        Data used to annotate.  If None, the image's data is used.  Optional.
    valfmt
        The format of the annotations inside the heatmap.  This should either
        use the string format method, e.g. "$ {x:.2f}", or be a
        `matplotlib.ticker.Formatter`.  Optional.
    textcolors
        A pair of colors.  The first is used for values below a threshold,
        the second for those above.  Optional.
    threshold
        Value in data units according to which the colors from textcolors are
        applied.  If None (the default) uses the middle of the colormap as
        separation.  Optional.
    **kwargs
        All other arguments are forwarded to each call to `text` used to create
        the text labels.
    """

    if not isinstance(data, (list, np.ndarray)):
        data = im.get_array()

    # Normalize the threshold to the images color range.
    if threshold is not None:
        threshold = im.norm(threshold)
    else:
        threshold = im.norm(data.max())/2.

    # Set default alignment to center, but allow it to be
    # overwritten by textkw.
    kw = dict(horizontalalignment="center",
              verticalalignment="center")
    kw.update(textkw)

    # Get the formatter in case a string is supplied
    if isinstance(valfmt, str):
        valfmt = mpl.ticker.StrMethodFormatter(valfmt)

    # Loop over the data and create a `Text` for each "pixel".
    # Change the text's color depending on the data.
    texts = []
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            kw.update(color=textcolors[int(im.norm(data[i, j]) > threshold)])
            text = im.axes.text(j, i, valfmt(data[i, j], None), **kw)
            texts.append(text)

    return texts

In [ ]:
fig, ax1 = plt.subplots(num="Correlation Map", figsize=(8,8))

cmap = mpl.cm.Spectral
bounds = np.linspace(-1, 1, 9)
norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend='both')
heatmap_df=df[["temperature_ESS301","temperature_ESS111", "temperature_ESS112", "temperature_ESS113", "wind_speed","tel_elevation","humidity"]][2000:-1]
heatmap_df["temperature_ESS111"]=heatmap_df["temperature_ESS111"]-heatmap_df["temperature_ESS112"]#["temperature_ESS301"]
heatmap_df["temperature_ESS112"]=heatmap_df["temperature_ESS112"]-heatmap_df["temperature_ESS112"]#["temperature_ESS301"]
heatmap_df["temperature_ESS113"]=heatmap_df["temperature_ESS113"]-heatmap_df["temperature_ESS112"]#["temperature_ESS301"]
heatmap_df["wind_speed"]=heatmap_df["wind_speed"]*np.cos(np.deg2rad(df["wind_direction"][2000:-1]-df["dome_az"][2000:-1]))
heatmap_df=heatmap_df[["temperature_ESS111", "temperature_ESS112", "temperature_ESS113", "wind_speed","tel_elevation","humidity"]]
heatmap_df["wind_speed"]=df["wind_speed"]*np.cos(np.deg2rad(df["wind_direction"]-df["dome_az"]))
im, cbar = heatmap(
    heatmap_df.corr(), 
    heatmap_df.columns, 
    heatmap_df.columns, 
    ax=ax1, 
    cbarlabel="Correlation", 
    cmap=cmap,
    norm=norm,
    cbar_kw=dict(
        ticks=np.linspace(-1, 1, 9), 
    )
)

annotate_heatmap(
    im=im,
    data=heatmap_df.corr(),
    fontsize="x-small"
)

fig.suptitle(f"Correlation Matrix - Day Obs {dayobs}")
plt.show()

In [ ]:
df_selected=df#[2500:-1]#[1600:2100]

camera_air_temp=df_selected["temperature_ESS111"]
m2_air_temp=df_selected["temperature_ESS112"]
m1m3_air_temp=df_selected["temperature_ESS113"]
outside_temp=df_selected["temperature_ESS301"]
TEA_temp=df_selected["temperature_TEA502"]
m2_glass_temp=df_selected["m2_glass_temp"]
m1m3_glass_temp=df_selected["m1m3_glass_temp"]
humidity=df_selected["humidity"]
louverE1=df_selected["louverE1"]
louverE2=df_selected["louverE2"]
louverB1=df_selected["louverB1"]
louverH1=df_selected["louverH1"]
louverH2=df_selected["louverH2"]
louverM1=df_selected["louverM1"]
az=df_selected["dome_az"]
el=df_selected["tel_elevation"]
z=np.sin(np.deg2rad(el))
wind=df_selected["wind_speed"]
wind_to_dome=wind*np.cos(np.deg2rad(df_selected["wind_direction"]-df_selected["dome_az"]))
#wind=wind
#wind_to_dome=wind_to_dome
shutter=df_selected["shutter"]
#louver=df_selected["louver"]

In [ ]:
l_open_e1=get_louver_closed_intervals(louverE1)
max_e1=l_open_e1[1]

l_open_e2=get_louver_closed_intervals(louverE2)
max_e2=l_open_e2[1]

l_open_b1=get_louver_closed_intervals(louverB1)
max_b1=l_open_b1[1]

l_open_m1=get_louver_closed_intervals(louverM1)
max_m1=l_open_m1[1]

l_open_h1=get_louver_closed_intervals(louverH1)
max_h1=l_open_h1[1]

l_open_h2=get_louver_closed_intervals(louverH2)
max_h2=l_open_h2[1]


In [ ]:
#find the time when dome was opened
closed_intervals=get_shutter_closed_intervals(shutter)
print(closed_intervals[0][1])
dome_open=(df.index==closed_intervals[0][1])
print(dome_open)
true_indices = [i for i, val in enumerate(dome_open) if val]

In [ ]:
z1=5.97*np.sin(np.deg2rad(el))+0.8*np.cos(np.deg2rad(el))
z2=1.62*np.sin(np.deg2rad(el))+3.6*np.cos(np.deg2rad(el))
z3=-1.38*np.sin(np.deg2rad(el))+4.08*np.cos(np.deg2rad(el))
print(np.mean(z1),np.mean(z2),np.mean(z3))

In [ ]:
df_selected=df[true_indices[0]-100:true_indices[0]+600]

camera_air_temp=df_selected["temperature_ESS111"]
m2_air_temp=df_selected["temperature_ESS112"]
m1m3_air_temp=df_selected["temperature_ESS113"]
outside_temp=df_selected["temperature_ESS301"]
TEA_temp=df_selected["temperature_TEA502"]
m2_glass_temp=df_selected["m2_glass_temp"]
m1m3_glass_temp=df_selected["m1m3_glass_temp"]
humidity=df_selected["humidity"]
louverE1=df_selected["louverE1"]
louverE2=df_selected["louverE2"]
louverB1=df_selected["louverB1"]
louverH1=df_selected["louverH1"]
louverH2=df_selected["louverH2"]
louverM1=df_selected["louverM1"]

el=df_selected["tel_elevation"]
z=np.sin(np.deg2rad(el))
wind=df_selected["wind_speed"]
wind_to_dome=wind*np.cos(np.deg2rad(df_selected["wind_direction"]-df_selected["dome_az"]))
#wind=wind
#wind_to_dome=wind_to_dome
shutter=df_selected["shutter"]
#louver=df_selected["louver"]

In [ ]:
dt=1/180

index_sample=df_selected[::2]#.rolling(window=3, center=True).mean()
m1m3_air_temp_sample=m1m3_air_temp[::2]#.rolling(window=3, center=True).mean()
m2_air_temp_sample=m2_air_temp[::2]#.rolling(window=3, center=True).mean()
camera_air_temp_sample=camera_air_temp[::2]#.rolling(window=3, center=True).mean()
outside_temp_sample=outside_temp[::2]#.rolling(window=3, center=True).mean()


velocities = np.diff(m1m3_air_temp_sample) / dt
v_m2=np.diff(m2_air_temp_sample)/dt
v_cam=np.diff(camera_air_temp_sample)/dt
v_out=np.diff(outside_temp_sample)/dt



fig,(ax,ax2,ax3,ax4)=plt.subplots(4,1,figsize=(10,7))
fig.suptitle(f"Temperature changing rate, {dayobs}")
closed_intervals=get_shutter_closed_intervals(shutter)

for start, end in closed_intervals:
    ax.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
    ax2.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
    ax3.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
    ax4.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
if closed_intervals:
    ax.axvspan(
            closed_intervals[0][0],
            closed_intervals[0][0],  # dummy zero-width span just for legend
            facecolor="lightblue",
            alpha=0.3,
            label="Shutter Closed",
    )
ax.plot(index_sample.index[:-1],v_out,c="C6")
ax.set_ylabel("ΔT_out/t [C/h]")
ax2.plot(index_sample.index[:-1],v_m2,c="C1")
ax2.set_ylim(bottom=np.nanmin(v_m2)-0.5, top=np.nanmax(v_m2)+0.5)
ax2.set_ylabel("ΔT_top/t [C/h]")
ax3.plot(index_sample.index[:-1],v_cam, c="C0")
ax3.set_ylim(bottom=np.nanmin(v_m2)-0.5, top=np.nanmax(v_m2)+0.5)
ax3.set_ylabel("ΔT_mid/t [C/h]")
ax4.plot(index_sample.index[:-1],velocities, c="C2")
ax4.set_ylim(bottom=np.nanmin(v_m2)-0.5, top=np.nanmax(v_m2)+0.5)
ax4.set_ylabel("ΔT_bottom/t [C/h]")

In [ ]:
fig, (ax,ax10,ax8,ax2,ax3,ax6, ax7) = plt.subplots(7,1,figsize=(10, 13), sharex=True)

fig.suptitle(f"Temperature , fwhm, height, wind, humidity {dayobs}")

#efd_shutter_results = await asyncio.create_task(query_efd_shutter(start, end))
#t1,t2=check_dome_open(start, end, **efd_shutter_results)
#print(t1,t2)
#if t2:
#    ax.axvline(t1, color="black", linestyle="--", linewidth=0.5, label="dome open starts")
#    ax.axvline(t2, color="black", linestyle="--", linewidth=0.5, label="dome open completes")
closed_intervals=get_shutter_closed_intervals(shutter)

for start, end in closed_intervals:
    ax.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
    ax10.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
    ax8.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
    ax2.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
    ax3.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
    ax6.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
    ax7.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")

if closed_intervals:
    ax.axvspan(
            closed_intervals[0][0],
            closed_intervals[0][0],  # dummy zero-width span just for legend
            facecolor="lightblue",
            alpha=0.3,
            label="Shutter Closed",
    )

ax.plot(camera_air_temp, color="C0", label="ESS:111 camera air Temperature")
ax.plot(m2_air_temp, color="C1", label="ESS:112 M2 air Temperature")
ax.plot(m1m3_air_temp, color="C2", label="ESS:113 M1M3 air Temperature")
#ax.plot(m2_glass_temp, color="C3", label="M2 glass Temperature",linestyle="--")
#ax.plot(m1m3_glass_temp, color="C4", label="M1M3 glass Temperature",linestyle="--")
#ax.plot(TEA_temp, color="C5", label="TEA502 Temperature")
ax.plot(outside_temp, color="C6", label="ESS:301 Outside Temperature")
ax.set_xlim(np.min(df_selected.index),np.max(df_selected.index))

#ax.plot(cbp_temp, color="C7", label="ESS:107 CBP Temperature")
#ax.grid(":", alpha=0.2)
ax.set_ylabel("Temperature [deg C]")
ax.legend(bbox_to_anchor=(1.35, 1),loc="upper right", fontsize="small")

ax10.plot(az, label="az")
ax10.set_ylabel("az")
ax4=ax10.twinx()
ax4.plot(el, color="grey", label="el")
ax10.set_xlim(np.min(df_selected.index),np.max(df_selected.index))
ax4.set_ylabel("el")
# Collect handles and labels from both axes
lines, labels = ax10.get_legend_handles_labels()
lines2, labels2 = ax4.get_legend_handles_labels()

# Create a combined legend
ax4.legend(lines + lines2, labels + labels2, bbox_to_anchor=(1.15, 1), loc='upper right', fontsize="small")
"""

annotate_bands(cdb_sub_table, ax8)
ax8.plot(cdb_sub_table.index,cdb_sub_table['fwhm_zenith_500nm'],  label='fwhm_zenith_500nm')
ax8.plot(cdb_sub_table.index,cdb_sub_table['dimm'], label='dimm')
ax8.plot(cdb_sub_table.index,cdb_sub_table['aos_fwhm'], label='aos_fwhm') #cdb_sub_table.index, 
ax8.legend(bbox_to_anchor=(1.25, 1.2),loc="upper right", fontsize="small")
ax8.set_xlim(np.min(df_selected.index),np.max(df_selected.index))
ax8.set_ylim(0,2.5)

ax8.set_ylabel("fwhm (arcsec)")

norm = plt.Normalize(np.min(m2_air_temp),np.max(m1m3_air_temp))
scatter_plot=ax2.scatter(z1.index, z1, c=m2_air_temp , cmap='coolwarm', norm=norm)
ax2.scatter(z2.index, z2, c=camera_air_temp, cmap='coolwarm', norm=norm)
ax2.scatter(z3.index, z3, c=m1m3_air_temp, cmap='coolwarm', norm=norm)
for start, end in closed_intervals:
    ax2.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
ax2.set_ylabel("height [m]")
#plt.colorbar(scatter_plot, label='Temperature (°C)', ax=ax2)
ax2.set_xlim(np.min(df_selected.index),np.max(df_selected.index))

cbar_ax = fig.add_axes([0.92, 0.45, 0.01, 0.1]) 
fig.colorbar(scatter_plot, label='Temperature (°C)',ax=ax2, pad=0.1, cax=cbar_ax)
"""
ax3.plot(wind, color="grey", label="Wind Speed", alpha=0.4)
ax3.set_ylabel("Wind Speed [m/s]")
ax5=ax3.twinx()
ax5.plot(wind_to_dome, color="black", alpha=0.4,label="Wind in direction of dome")
ax5.set_ylabel("in direction of shutter [m/s]")
ax3.set_xlim(np.min(df_selected.index),np.max(df_selected.index))
# Collect handles and labels from both axes
lines, labels = ax3.get_legend_handles_labels()
lines2, labels2 = ax5.get_legend_handles_labels()

# Create a combined legend
ax5.legend(lines + lines2, labels + labels2, loc='upper right', fontsize="small")


ax6.plot(humidity, color="grey", label="humidity", alpha=0.4)
ax6.set_ylabel("humidity [%]")
ax6.set_xlim(np.min(df_selected.index),np.max(df_selected.index))

"""
ax7.plot(louverE1, label=f"E1, {max_e1:.0f}%")
ax7.plot(louverE2, label=f"E2, {max_e2:.0f}%")
ax7.plot(louverB1, label=f"B1, {max_b1:.0f}%")
ax7.plot(louverM1, label=f"M1, {max_m1:.0f}%")
ax7.plot(louverH1, label=f"H1, {max_h1:.0f}%")
ax7.plot(louverH2, label=f"H2, {max_h2:.0f}%")
ax7.set_ylabel("louvers")
ax7.legend(bbox_to_anchor=(1.15, 1),loc="upper right", fontsize="small")
ax7.set_xlim(np.min(df_selected.index),np.max(df_selected.index))
"""

fig.autofmt_xdate()
plt.show()

In [ ]:
fig, (ax,ax8,ax2,ax3,ax6, ax7) = plt.subplots(6,1,figsize=(10, 13))

fig.suptitle(f"Temperature , fwhm, height, wind, humidity {dayobs}")

#efd_shutter_results = await asyncio.create_task(query_efd_shutter(start, end))
#t1,t2=check_dome_open(start, end, **efd_shutter_results)
#print(t1,t2)
#if t2:
#    ax.axvline(t1, color="black", linestyle="--", linewidth=0.5, label="dome open starts")
#    ax.axvline(t2, color="black", linestyle="--", linewidth=0.5, label="dome open completes")
closed_intervals=get_shutter_closed_intervals(shutter)

for start, end in closed_intervals:
    ax.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
if closed_intervals:
    ax.axvspan(
            closed_intervals[0][0],
            closed_intervals[0][0],  # dummy zero-width span just for legend
            facecolor="lightblue",
            alpha=0.3,
            label="Shutter Closed",
    )

ax.plot(camera_air_temp, color="C0", label="ESS:111 camera air Temperature")
ax.plot(m2_air_temp, color="C1", label="ESS:112 M2 air Temperature")
ax.plot(m1m3_air_temp, color="C2", label="ESS:113 M1M3 air Temperature")
#ax.plot(m2_glass_temp, color="C3", label="M2 glass Temperature",linestyle="--")
#ax.plot(m1m3_glass_temp, color="C4", label="M1M3 glass Temperature",linestyle="--")
#ax.plot(TEA_temp, color="C5", label="TEA502 Temperature")
ax.plot(outside_temp, color="C6", label="ESS:301 Outside Temperature")

#ax.plot(cbp_temp, color="C7", label="ESS:107 CBP Temperature")
#ax.grid(":", alpha=0.2)
ax.set_ylabel("Temperature [deg C]")
ax.legend(bbox_to_anchor=(1.3, 1),loc="upper right", fontsize="small")
"""
ax8.plot(cdb_sub_table.index, cdb_sub_table['fwhm_zenith_500nm'],  label='fwhm_zenith_500nm')
ax8.plot(cdb_sub_table.index, cdb_sub_table['dimm'], label='dimm')
ax8.plot(cdb_sub_table.index, cdb_sub_table['aos_fwhm'], label='aos_fwhm')
ax8.set_ylabel('FWHM (arcsec)')
ax8.set_title('Image Quality Measurements')
ax8.legend(bbox_to_anchor=(1.2, 1),loc="upper right", fontsize="small")

norm = plt.Normalize(np.min(m2_air_temp),np.max(m1m3_air_temp))
scatter_plot=ax2.scatter(z1.index, z1, c=m2_air_temp , cmap='coolwarm', norm=norm)
ax2.scatter(z2.index, z2, c=camera_air_temp, cmap='coolwarm', norm=norm)
ax2.scatter(z3.index, z3, c=m1m3_air_temp, cmap='coolwarm', norm=norm)
for start, end in closed_intervals:
    ax2.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
ax2.set_ylabel("height [m]")
#plt.colorbar(scatter_plot, label='Temperature (°C)', ax=ax2)


ax3.plot(m2_air_temp-outside_temp, color="C6", label="m2_air-outside")
ax3.plot(camera_air_temp-m1m3_air_temp, color="C2", label="camera_air-m1m3_air")
ax3.plot(m2_air_temp-camera_air_temp, color="C1", label="m2_air-camera_air")

ax3.set_ylabel("Temperature difference [C]")
ax3.legend(bbox_to_anchor=(1.2, 1),loc="upper right", fontsize="small")


ax6.plot(humidity, color="grey", label="humidity", alpha=0.4)
ax6.set_ylabel("humidity [%]")


ax7.plot(wind, color="grey", label="Wind Speed", alpha=0.4)
ax7.set_ylabel("Wind Speed [m/s]")
ax7.legend(loc="lower left", fontsize="small")
ax5=ax7.twinx()
ax5.plot(wind_to_dome, color="black", alpha=0.4,label="Wind in direction of dome")
ax5.set_ylabel("in direction of shutter [m/s]")
"""
fig.autofmt_xdate()
plt.show()

In [ ]:
fig, (ax) = plt.subplots(1,1,figsize=(9, 4))
ax.plot(z1, color="C1")
ax.plot(z2, color="C0")
ax.plot(z3, color="C2")
ax1=ax.twinx()
ax1.plot(el, color="grey", alpha=0.3)

In [ ]:
fig, (ax) = plt.subplots(1,1,figsize=(15, 3))
fig.suptitle(f"Temperature flow in height, {dayobs}")

norm = plt.Normalize(np.min(m2_air_temp),np.max(m1m3_air_temp))
scatter_plot=ax.scatter(z1.index, z1, c=m2_air_temp , cmap='coolwarm', norm=norm)
ax.scatter(z2.index, z2, c=camera_air_temp, cmap='coolwarm', norm=norm)
ax.scatter(z3.index, z3, c=m1m3_air_temp, cmap='coolwarm', norm=norm)
for start, end in closed_intervals:
    ax.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
ax.set_ylabel("height [m]")
plt.colorbar(scatter_plot, label='Temperature (°C)')

In [ ]:
fig, (ax,ax2,ax3,ax6) = plt.subplots(4,1,figsize=(10, 9))

fig.suptitle(f"Temperature-air near M2 temp (112),Telescope Elevation, and Wind, {dayobs}")

closed_intervals=get_shutter_closed_intervals(shutter)

for start, end in closed_intervals:
    ax.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
if closed_intervals:
    ax.axvspan(
            closed_intervals[0][0],
            closed_intervals[0][0],  # dummy zero-width span just for legend
            facecolor="lightblue",
            alpha=0.3,
            label="Shutter Closed",
    )
h_factor=(z1*humidity)/np.mean(z1*humidity)

#ax.plot(h_factor-z2/8, color="C0", label="z2&humidity factor",linestyle="--")
#ax.plot(h_factor+z3/8, color="C2", label="z3&humidity factor",linestyle="--")
ax.plot(h_factor, color="grey", label="humidity factor")
ax.plot((camera_air_temp-m2_air_temp), color="C0", label="ESS:111 camera air Temperature")
ax.plot(m2_air_temp-m2_air_temp, color="C1", label="ESS:112 M2 air Temperature")
ax.plot((m1m3_air_temp-m2_air_temp), color="C2", label="ESS:113 M1M3 air Temperature")
#ax.plot(m2_glass_temp-m1m3_air_temp, color="C3", label="M2 glass Temperature",linestyle="--")
#ax.plot(m1m3_glass_temp-m1m3_air_temp, color="C4", label="M1M3 glass Temperature", linestyle="--")
#ax.plot(TEA_temp-m1m3_air_temp, color="C5", label="TEA502 Temperature")
#ax.plot(outside_temp, color="C4", label="ESS:301 Outside Temperature")

#ax.grid(":", alpha=0.2)
ax.set_ylabel("Temperature [deg C]")
ax.legend(bbox_to_anchor=(1.3, 1),loc="upper right", fontsize="small")

#ax2 = ax.twinx()
# ax2.plot(el, color="C5", label="Telescope Elevation", alpha=0.4)
# ax2.set_ylabel("Elevation Angle [deg]")
# ax2.legend(loc="lower left", fontsize="small")
# ax4=ax2.twinx()
# ax4.plot(z,color="black", label="Height z", alpha=0.4)
# ax4.set_ylabel("El->z")

# ax3.plot(camera_air_temp/(h_factor), color="C0", label="ESS:111 camera air Temperature/h_factor")
# ax3.plot(m2_air_temp/(h_factor), color="C1", label="ESS:112 M2 air Temperature/h_factor")
# ax3.plot(m1m3_air_temp/(h_factor), color="C2", label="ESS:113 M1M3 air Temperature/h_factor")


# ax6.plot(camera_air_temp, color="C0", label="ESS:111 camera air Temperature")
# ax6.plot(m2_air_temp, color="C1", label="ESS:112 M2 air Temperature")
# ax6.plot(m1m3_air_temp, color="C2", label="ESS:113 M1M3 air Temperature")


norm = plt.Normalize(np.min(m2_air_temp),np.max(m1m3_air_temp))
scatter_plot=ax2.scatter(z1.index, z1, c=m2_air_temp , cmap='coolwarm', norm=norm)
ax2.scatter(z2.index, z2, c=camera_air_temp, cmap='coolwarm', norm=norm)
ax2.scatter(z3.index, z3, c=m1m3_air_temp, cmap='coolwarm', norm=norm)
for start, end in closed_intervals:
    ax2.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
    ax3.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
ax2.set_ylabel("height [m]")


ax3.plot(m2_air_temp, color="C1", label="ESS:112 top air Temperature")
ax3.plot(camera_air_temp, color="C0", label="ESS:111 mid air Temperature")
ax3.plot(m1m3_air_temp, color="C2", label="ESS:113 bottom air Temperature")
ax3.plot(outside_temp, color="C6", label="ESS:301 Outside Temperature")

ax3.set_ylabel("Temperature [deg C]")
ax3.legend(bbox_to_anchor=(1.3, 1),loc="upper right", fontsize="small")

ax6.plot(humidity, color="grey", label="humidity", alpha=0.4)
ax6.set_ylabel("Humidity [%]")
ax6.legend(loc="lower left", fontsize="small")
fig.autofmt_xdate()
plt.show()

In [ ]:
df_selected=df#[:2400]#[2300:-1]
camera_air_temp=df_selected["temperature_ESS111"]
m2_air_temp=df_selected["temperature_ESS112"]
m1m3_air_temp=df_selected["temperature_ESS113"]
outside_temp=df_selected["temperature_ESS301"]
TEA_temp=df_selected["temperature_TEA502"]
m2_glass_temp=df_selected["m2_glass_temp"]
m1m3_glass_temp=df_selected["m1m3_glass_temp"]
humidity=df_selected["humidity"]

el=df_selected["tel_elevation"]
z=np.sin(np.deg2rad(el))
wind=df_selected["wind_speed"]
wind_to_dome=wind*np.cos(np.deg2rad(df_selected["wind_direction"]-df_selected["dome_az"]))
#wind=wind
#wind_to_dome=wind_to_dome
shutter=df_selected["shutter"]
louver=df_selected["louver"]

In [ ]:
fig, (ax,ax2,ax3,ax6) = plt.subplots(4,1,figsize=(10, 9))

fig.suptitle(f"Temperature-outside temperature, {dayobs}")

closed_intervals=get_shutter_closed_intervals(shutter)

for start, end in closed_intervals:
    ax.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
if closed_intervals:
    ax.axvspan(
            closed_intervals[0][0],
            closed_intervals[0][0],  # dummy zero-width span just for legend
            facecolor="lightblue",
            alpha=0.3,
            label="Shutter Closed",
    )

ax.plot(m2_air_temp-outside_temp, color="C1", label="ESS:112 top air Temperature")
ax.plot(camera_air_temp-outside_temp, color="C0", label="ESS:111 mid air Temperature")
ax.plot(m1m3_air_temp-outside_temp, color="C2", label="ESS:113 bottom air Temperature")
#ax.plot(m2_glass_temp-outside_temp-, color="C3", label="M2 glass Temperature",linestyle="--")
h_factor=(humidity)/np.mean(humidity)

#ax.plot(h_factor-z2/8, color="C0", label="z2&humidity factor",linestyle="--")
#ax.plot(h_factor+z3/8, color="C2", label="z3&humidity factor",linestyle="--")
#ax.plot(h_factor, color="grey", label="humidity factor")
#ax.plot(outside_temp-TEA_temp, color="C4", label="TEA502 Temperature")
#ax.plot(outside_temp, color="C4", label="ESS:301 Outside Temperature")

#ax.grid(":", alpha=0.2)
ax.set_ylabel("Temperature-outside [deg C]")
ax.legend(bbox_to_anchor=(1.3, 1),loc="upper right", fontsize="small")



norm = plt.Normalize(np.min(m2_air_temp),np.max(m1m3_air_temp))
scatter_plot=ax2.scatter(z1.index, z1, c=m2_air_temp , cmap='coolwarm', norm=norm)
ax2.scatter(z2.index, z2, c=camera_air_temp, cmap='coolwarm', norm=norm)
ax2.scatter(z3.index, z3, c=m1m3_air_temp, cmap='coolwarm', norm=norm)
for start, end in closed_intervals:
    ax2.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
    ax3.axvspan(start,end,facecolor="lightblue", alpha=0.3, zorder=0, label="_shutter_closed")
ax2.set_ylabel("height [m]")


ax3.plot(m2_air_temp, color="C1", label="ESS:112 top air Temperature")
ax3.plot(camera_air_temp, color="C0", label="ESS:111 mid air Temperature")
ax3.plot(m1m3_air_temp, color="C2", label="ESS:113 bottom air Temperature")
ax3.plot(outside_temp, color="C6", label="ESS:301 Outside Temperature")

ax3.set_ylabel("Temperature [deg C]")
ax3.legend(bbox_to_anchor=(1.3, 1),loc="upper right", fontsize="small")

ax6.plot(humidity, color="grey", label="humidity", alpha=0.4)
ax6.set_ylabel("Humidity [%]")
ax6.legend(loc="lower left", fontsize="small")
fig.autofmt_xdate()
plt.show()